# KVO_08 — Satellite prediction of Musanga canopy fraction

**Five sites:** MOS_1_2, Yoko, Yangambi, Baego, Luki.

**Phase 1: data-preparation/validation.** The first runnable cells audit existing NB04 products without changing them. The NB04 production manifest documents the four-site final frozen model, uint16 0–10000 probability and 65535 NoData. Luki is an external site. The four current NB04 maps may be annotation-ROI restricted; their *pre-vector-ROI-mask* backups have broader coverage but must not be assumed full mosaic until audited.

**CRS rule:** keep UAV source pixels in their native projected CRS (Luki EPSG:32733; other sites verified individually), and later construct per-site target grids aligned to a chosen **actual Sentinel-2 10 m projection + transform**. Satellite embeddings and S2 are *not assumed* co-aligned; verify exact grid transforms first. No reprojection is performed by the audit.

**Run Blocks 00–02 now (CPU).** They only read metadata and small samples of probability maps; they never run U-Net, change any raster, or download satellites. All results go to `KVO_work/KVO_08_satellite_fraction/validation/`.

In [ ]:
# BLOCK 00 — canonical site paths; CPU-only, read-only inspection
from pathlib import Path
import json, math, shutil
import numpy as np
import pandas as pd
import rasterio
from rasterio.windows import Window
from IPython.display import display
from google.colab import drive

MY = Path("/content/drive/MyDrive")
if not MY.exists(): drive.mount("/content/drive")
ROOT = MY / "Colab Notebooks" / "Musanga"
GEO = ROOT / "data" / "Geo_Ref"
KVO04 = ROOT / "KVO_work" / "KVO_04_final_multisite"
MAPS = KVO04 / "probability_maps"
OUT = ROOT / "KVO_work" / "KVO_08_satellite_fraction"
VALIDATION = OUT / "validation"
VALIDATION.mkdir(parents=True, exist_ok=True)

RGB = {
    "MOS_1_2": GEO / "MOS_1_2_kvo.tif",
    "Yoko": GEO / "Yoko_kvo_georef.tif",
    "Yangambi": GEO / "Yangambi_georef.tif",
    "Baego": GEO / "Baego_georef.tif",
    "Luki": GEO / "Luki.tif",
}
CURRENT = {site: MAPS / f"{site}_musanga_probability_uint16.tif" for site in RGB}
BACKUP = {
    site: MAPS / f"{site}_musanga_probability_uint16.pre_vector_roi_mask.tif"
    for site in RGB if site != "Luki"
}
FULL_LUKI = MAPS / "Luki_musanga_probability_uint16_FULL.tif"

NODATA = 65535
SCALE = 10000
SITES = list(RGB)
print("Validation outputs:", VALIDATION)
print("Sites:", ", ".join(SITES))
print("Luki FULL map present:", FULL_LUKI.exists())
print("No inference, overwrites, reprojecting, or downloads in Blocks 00–02.")

In [ ]:
# BLOCK 01 — Native-CRS metadata and probability schema checks (READ-ONLY)
# Candidate types are recorded explicitly. 'pre_ROI_backup' is NOT assumed full coverage.
import warnings
def meta(path):
    with rasterio.open(path) as ds:
        return {
            "width": ds.width, "height": ds.height, "crs": str(ds.crs),
            "transform": tuple(ds.transform), "bounds": tuple(ds.bounds),
            "px_x_m": abs(ds.transform.a), "px_y_m": abs(ds.transform.e),
            "dtype": ds.dtypes[0], "bands": ds.count, "nodata": ds.nodata,
            "tags": ds.tags(),
            "bbox_ha": ds.width*ds.height*abs(
                ds.transform.a*ds.transform.e-ds.transform.b*ds.transform.d
            )/10000,
        }

rows = []
candidates = []
for site in SITES:
    rgb_path = RGB[site]
    base = {"site":site,"rgb_path":str(rgb_path),
            "rgb_exists":rgb_path.is_file(),
            "current_exists":CURRENT[site].is_file()}
    if not rgb_path.is_file():
        rows.append({**base,"status":"MISSING_RGB"})
        continue
    rgb_meta = meta(rgb_path)
    assert rgb_meta["crs"]!="None",f"{site}: RGB CRS is undefined"
    base.update(rgb_crs=rgb_meta["crs"],
                gsd_x_m=rgb_meta["px_x_m"],gsd_y_m=rgb_meta["px_y_m"],
                rgb_bands=rgb_meta["bands"],
                width=rgb_meta["width"],height=rgb_meta["height"],
                rgb_bbox_ha=rgb_meta["bbox_ha"])
    candidate_paths = [("current",CURRENT[site])]
    if site in BACKUP: candidate_paths.append(("pre_ROI_backup",BACKUP[site]))
    if site=="Luki": candidate_paths.append(("full_mosaic",FULL_LUKI))
    for kind,p in candidate_paths:
        item={**base,"candidate_kind":kind,"probability_path":str(p),
              "probability_exists":p.is_file()}
        if not p.is_file():
            item["status"]="MISSING_PROBABILITY"
            rows.append(item)
            continue
        pm=meta(p)
        same_grid=(pm["width"]==rgb_meta["width"] and
                   pm["height"]==rgb_meta["height"] and
                   pm["crs"]==rgb_meta["crs"] and
                   np.allclose(pm["transform"],rgb_meta["transform"],rtol=0,atol=1e-6))
        schema_ok=(pm["bands"]==1 and pm["dtype"]=="uint16" and
                   pm["nodata"]==NODATA)
        item.update(prob_crs=pm["crs"],prob_dtype=pm["dtype"],
                    prob_nodata=pm["nodata"],same_uav_grid=same_grid,
                    schema_ok=schema_ok,coverage_tag=pm["tags"].get("COVERAGE",""),
                    model_checkpoint=pm["tags"].get("MODEL_CHECKPOINT",""),
                    probability_file_GB=round(p.stat().st_size/1e9,3),
                    status="SCHEMA_OK" if same_grid and schema_ok else "SCHEMA_ERROR")
        rows.append(item)
        candidates.append((site,kind,p,item["status"]))
audit=pd.DataFrame(rows)
audit.to_csv(VALIDATION/"01_reference_metadata.csv",index=False)
display(audit[["site","candidate_kind","probability_exists","rgb_crs",
               "rgb_bbox_ha","same_uav_grid","schema_ok","coverage_tag",
               "status"]].round(2))
assert any(z[3]=="SCHEMA_OK" for z in candidates),"No valid probability candidate found."
print("IMPORTANT: current and backup files have identical raster *dimensions*;")
print("this says nothing about their actual valid-pixel coverage.")

In [ ]:
# BLOCK 02 — Stratified same-location probability coverage comparison
# Fast read-only sample; NOT an exact entire-mosaic valid-area computation.
# Sample 12 x 12 source-coordinate locations with 128-px windows each.
GRID_N = 12
SAMPLE_PX = 128
sample_rows=[]
for site in SITES:
    if not RGB[site].exists():continue
    site_candidates=[(kind,path) for s,kind,path,status in candidates
                     if s==site and status=="SCHEMA_OK"]
    if not site_candidates:continue
    with rasterio.open(RGB[site]) as src:
        locations=[]
        for yf in (np.arange(GRID_N)+.5)/GRID_N:
            for xf in (np.arange(GRID_N)+.5)/GRID_N:
                row=int(yf*src.height);col=int(xf*src.width)
                row=max(0,min(row-SAMPLE_PX//2,src.height-SAMPLE_PX))
                col=max(0,min(col-SAMPLE_PX//2,src.width-SAMPLE_PX))
                locations.append(Window(col,row,min(SAMPLE_PX,src.width),min(SAMPLE_PX,src.height)))
        area_ha=src.width*src.height*abs(
            src.transform.a*src.transform.e-src.transform.b*src.transform.d)/1e4
    for kind,path in site_candidates:
        n=0;valid_total=0;invalid_encoding=0;mean_sum=0.0
        counts=[]
        with rasterio.open(path) as ds:
            for w in locations:
                a=ds.read(1,window=w)
                good=(a!=NODATA)&(a<=SCALE)
                invalid_encoding+=int(((a>NODATA)|( (a>SCALE)&(a!=NODATA))).sum())
                n+=a.size
                valid_total+=int(good.sum())
                if good.any():mean_sum+=float(a[good].astype(np.float64).sum())
                counts.append(float(good.mean()))
        frac=valid_total/max(n,1)
        sample_rows.append({
            "site":site,"candidate_kind":kind,"path":str(path),
            "sampled_pixels":n,"sample_valid_pixels":valid_total,
            "sample_valid_fraction":frac,
            "sample_mean_probability":mean_sum/valid_total/SCALE if valid_total else np.nan,
            "sample_invalid_encoding_pixels":invalid_encoding,
            "approx_valid_ha_NOT_EXACT":area_ha*frac,
            "sample_window_valid_fraction_min":min(counts),
            "sample_window_valid_fraction_max":max(counts),
        })
coverage=pd.DataFrame(sample_rows)
coverage.to_csv(VALIDATION/"02_stratified_coverage.csv",index=False)
display(coverage[["site","candidate_kind","sample_valid_fraction",
                  "sample_mean_probability","sample_window_valid_fraction_min",
                  "sample_window_valid_fraction_max","sample_invalid_encoding_pixels"]].round(4))
assert (coverage.sample_invalid_encoding_pixels==0).all(),"Out-of-range probabilities found."
print("Sample coverage is indicative only. ROI masks, orthomosaic edges and clustering")
print("mean we must NOT use the approximate area as exact valid-imagery area.")

# Select the broadest available candidate per site for **inspection only**.
# Final acceptance requires rigorous coverage analysis and source provenance.
rank={"full_mosaic":0,"pre_ROI_backup":1,"current":2}
coverage["priority"]=coverage.candidate_kind.map(rank)
selected=(coverage.sort_values(["site","sample_valid_fraction","priority"],
                              ascending=[True,False,True])
                  .drop_duplicates("site").copy())
selected["audit_decision"]="INSPECT_REQUIRED"
selected["reason"]="Coverage and full-mosaic provenance not yet independently certified"
selected.to_csv(VALIDATION/"02_candidate_selection_for_review.csv",index=False)
print("\nRECOMMENDED CANDIDATES FOR COVERAGE REVIEW (not accepted as FULL):")
display(selected[["site","candidate_kind","sample_valid_fraction","audit_decision"]].round(4))
missing=set(SITES)-set(selected.site)
if missing:print("MISSING candidate sites:",sorted(missing))
print("DATA GATE: full-mosaic status and satellite grid alignment remain unresolved.")
print("No satellite target grids or training tables have been generated.")

## 02 — Research design and fixed modelling contract

### UAV reference target
- Create one **5 m globally aligned UTM reference grid per site** from *full-coverage* UAV maps, with pixel area weighting where necessary.
- Maintain **valid UAV observation fraction** and **mean Musanga probability** as separate bands. Derive cover-fraction alternatives based on calibrated or fixed U-Net probability thresholds; never treat unmapped pixels as zero.
- Consider crown labels / Luki inventory for independent UAV-prediction calibration and audit. Thresholds must not be optimized using held-out satellite sites.
- Require a minimum observed high-resolution area per satellite grid cell (initial proposal ≥90%; sensitivity: 70%, 80%, 95%).
- Use strictly aligned support grids at 10, 30, 50, 75 and 100 m, all integer multiples of 5 m. Do not interpolate a 10 m satellite measurement into novel 10 m independent observations.

### Satellite feature families (independently ablated)
- **S00** geographic/climatological null baseline only; avoid site identifiers as predictors in site-held-out transfer.
- **S01** Sentinel-2 SR: B2/B3/B4/B5/B6/B7/B8/B8A/B11/B12 and NDVI, EVI, NDRE, NBR, NDMI; use correct resampling of native 20 m bands.
- **S02** Sentinel-2 quality-filtered seasonal percentiles and cloud-free acquisition counts, matched to UAV acquisition epoch.
- **S03** Sentinel-2 monthly/quarterly trajectories, phenological amplitudes and long-term trend; explicitly check missing periods and date leakage.
- **E01** Earth-observation embeddings (e.g. annual Sentinel-2-based embeddings), with provenance, source year and spatial support recorded.
- **F01** spectral + temporal; **F02** embeddings + spectral; **F03** all combined.
- Optional later: Sentinel-1, terrain, forest disturbance history. Keep separate so contribution is identifiable.

### Resolution experiment
Model 10, 30, 50, 75, 100 m *support* with the same site holdouts and comparable coverage gates. Distinguish (i) physical aggregation of UAV fraction, (ii) satellite native spatial resolution, (iii) smoothing/resampling. A 10 m multispectral feature stack contains native 20 m bands and must be labelled accordingly.

### Validation design
**Primary:** five leave-one-site-out folds, training/tuning on four sites; avoid using Luki for any model choice in its held-out fold.
**Secondary:** buffered spatial-block cross-validation within the training sites, blocks substantially larger than the 100 m response grid (test 500 m / 1 km based on autocorrelation).
**No leakage:** tune preprocessing, feature selection, threshold calibration and hyperparameters inside each training fold. Sample weights should avoid dominance by large mosaics and spatially autocorrelated pixels.

### Algorithms
Reference baselines: constant mean, ridge/elastic net and random forest. Main models: HistGradientBoosting or LightGBM plus a regularized embedding model. Optional spatial CNN only after tabular transfer baseline is established. Predict bounded fractions via suitable link/loss or calibration.

### Metrics and science
Primary: site-held-out MAE and RMSE (fraction units), calibration bias and R² relative to within-site variance. Secondary: Spearman ranking, occurrence detection AUC/AP where prevalence is relevant, abundance-stratified error, quality-/coverage-stratified error and bootstrap confidence intervals by spatial block. Report site-by-site and macro averages; do not use random pixel-split scores as external transfer estimates.

### Expected outputs
- Reproducible coverage-checked 5 m UAV reference grids, plus aggregate target grids at all five scales.
- Five-site harmonized spectral/temporal/embedding feature tables and provenance.
- Predeclared experiment matrix and leak-proof fold manifests.
- LO(S)O benchmark, feature ablations, support-scale curves, SHAP/permutation-importance checks on held-out data.
- Final full-predictor model and uncertainty rasters, then ecological tests against disturbance/succession.

In [ ]:
# 03 — Prespecified experiment registry (scaffold; no model fitting)
resolutions=[10,30,50,75,100]
feature_families=["S01_S2_static","S02_S2_seasonal","S03_S2_trajectory",
                  "E01_embeddings","F01_spectral_temporal","F02_embedding_spectral","F03_all"]
algorithms=["ridge","random_forest","hist_gradient_boosting"]
experiments=pd.DataFrame(
    [(r,f,a) for r in resolutions for f in feature_families for a in algorithms],
    columns=["resolution_m","feature_family","algorithm"]
)
experiments["status"]="NOT_RUN"
experiments.to_csv(OUT/"experiment_registry.csv",index=False)
print("Registered experiments:",len(experiments))
print("× 5 independent site holdouts; not fitted until data QA passes.")
display(experiments.head(12))